# Processing raw velocity data

Computes velocity from `treadmillPosition` in the intermediate behavior JSON
(output of `process_tdml_behavior_data.py`) and saves it as
`[[time_s, velocity], ...]` pairs to `filtered_velocity.json`.

**Pipeline position:**
```
raw .tdml
  → process_tdml_behavior_data.py  → <session>.json   (treadmillPosition pairs)
  → this notebook                  → filtered_velocity.json  ([[time_s, vel], ...])
  → BehaviorData.resample_to_imaging()               (uniform imaging-rate grid)
```

**What is saved:** raw event-driven velocity with per-event timestamps.  
Gaussian smoothing is shown below for inspection only and is **not** written
to disk — apply it after resampling to the imaging grid in the analysis workflow.

**"filtered"** in the filename now refers to optional outlier removal (NaN-drop),
not Gaussian smoothing.

In [ ]:
import json
from os.path import join

import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter


## Configuration

Set `behavior_json` to the `.json` file produced by `process_tdml_behavior_data.py`  
and `output_dir` to the `.sima/behavior/` folder where `filtered_velocity.json` will be saved.

In [ ]:
# Path to the intermediate JSON from process_tdml_behavior_data.py
behavior_json = "/data2/gergely/BehaviorData/<mouse_id>/<session>.json"

# Path to the .sima/behavior/ folder where filtered_velocity.json will be written
output_dir = "/data2/gergely/invivo_DATA/sleep/<mouse_id>/TSeries-<...>.sima/behavior"

## Load and compute velocity

Velocity at each BehaviorMate event = Δnormalized_position / Δtime_s.  
The timestamp for each velocity sample is the start of the inter-event interval.

In [ ]:
with open(behavior_json) as f:
    data = json.load(f)

tp = np.array(data["treadmillPosition"])   # shape (N, 2): [time_s, norm_position]
dt = np.diff(tp[:, 0])                     # inter-event durations
dp = np.diff(tp[:, 1])                     # position changes (normalized)
velocity = dp / dt                          # shape (N-1,)
times = tp[:-1, 0]                          # timestamp = start of interval

velocity_ts = np.column_stack([times, velocity])   # shape (N-1, 2)

print(f"events:           {len(velocity_ts)}")
print(f"time range:       {times[0]:.2f} s → {times[-1]:.2f} s")
print(f"recordingDuration:{data['recordingDuration']} s")
print(f"velocity range:   {velocity.min():.4f} → {velocity.max():.4f} (normalized pos / s)")

## Inspect velocity

Zoom and pan to locate encoder-glitch spikes.  Hover over any point to read its **event index** — that is the
value to put into `outlier_slices` in the next cell.

The σ=2 Gaussian trace is hidden by default; click it in the legend to
toggle it on.  Neither trace affects what gets saved.


In [ ]:
smoothed = gaussian_filter(velocity.astype(float), sigma=2)
indices = np.arange(len(times))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times,
    y=velocity,
    mode="lines",
    name="raw velocity",
    line=dict(width=0.8, color="steelblue"),
    customdata=indices,
    hovertemplate=(
        "<b>idx: %{customdata}</b><br>"
        "time: %{x:.3f} s<br>"
        "vel: %{y:.5f}<extra></extra>"
    ),
))

fig.add_trace(go.Scatter(
    x=times,
    y=smoothed,
    mode="lines",
    name="σ=2 preview (not saved)",
    line=dict(width=1.5, color="coral"),
    customdata=indices,
    hovertemplate=(
        "<b>idx: %{customdata}</b><br>"
        "time: %{x:.3f} s<br>"
        "vel (σ=2): %{y:.5f}<extra></extra>"
    ),
    visible="legendonly",
))

fig.update_layout(
    height=450,
    xaxis_title="Time (s)",
    yaxis_title="Velocity (norm. pos / s)",
    hovermode="closest",
    title="Hover a spike → read idx → paste into outlier_slices",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()


## Outlier removal (optional)

If the plot above shows spikes caused by encoder glitches, mark the
corresponding event-index ranges in `outlier_slices` below.  Those events
are dropped from the time series before saving; `resample_to_imaging()` will
bridge the gap via linear interpolation.

Leave `outlier_slices = []` to skip this step.

In [ ]:
# Add (start, stop) index pairs for events to remove, e.g. [(13400, 13409)].
outlier_slices = [
    # (start, stop),
]

if outlier_slices:
    mask = np.ones(len(velocity), dtype=bool)
    for start, stop in outlier_slices:
        mask[start:stop] = False
    velocity_ts = velocity_ts[mask]
    print(f"Dropped {(~mask).sum()} events; {len(velocity_ts)} remaining.")
else:
    print("No outlier slices defined — all events retained.")

## Rejection confirmation

Inspect before saving. The cleaned trace previews the linear bridge that
`resample_to_imaging()` will produce — same `np.interp` anchored on the
retained events, evaluated at the original event times.  Red shading marks
each dropped index range.  If `outlier_slices` is empty the two traces
coincide (nothing rejected).


In [ ]:
# Evaluate post-rejection velocity_ts at the ORIGINAL event times —
# matches the linear interpolation used by resample_to_imaging().
bridged = np.interp(times, velocity_ts[:, 0], velocity_ts[:, 1])
indices = np.arange(len(times))

fig = go.Figure()

# original trace (faint) — retains visible spikes for comparison
fig.add_trace(go.Scatter(
    x=times,
    y=velocity,
    mode="lines",
    name="original (with spikes)",
    line=dict(width=0.6, color="steelblue"),
    opacity=0.3,
    customdata=indices,
    hovertemplate=(
        "<b>idx: %{customdata}</b><br>"
        "time: %{x:.3f} s<br>"
        "vel (original): %{y:.5f}<extra></extra>"
    ),
))

# cleaned trace — linear bridge over dropped events
fig.add_trace(go.Scatter(
    x=times,
    y=bridged,
    mode="lines",
    name="cleaned (interp preview)",
    line=dict(width=1.2, color="steelblue"),
    customdata=indices,
    hovertemplate=(
        "<b>idx: %{customdata}</b><br>"
        "time: %{x:.3f} s<br>"
        "vel (cleaned): %{y:.5f}<extra></extra>"
    ),
))

# shade each rejected region
for start, stop in outlier_slices:
    fig.add_vrect(
        x0=times[start],
        x1=times[min(stop, len(times)) - 1],
        fillcolor="red", opacity=0.15, layer="below", line_width=0,
    )

fig.update_layout(
    height=400,
    xaxis_title="Time (s)",
    yaxis_title="Velocity (norm. pos / s)",
    hovermode="closest",
    title="Rejection preview — red = dropped; cleaned trace shows linear bridge",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.show()


## Save

Writes `[[time_s, velocity], ...]` pairs — the format expected by
`BehaviorData.resample_to_imaging()`.

In [ ]:
output_path = join(output_dir, "filtered_velocity.json")
with open(output_path, "w") as f:
    json.dump(velocity_ts.tolist(), f, indent=4)
print(f"Saved {len(velocity_ts)} events → {output_path}")

## Verify

In [ ]:
with open(output_path) as f:
    loaded = np.array(json.load(f))

assert loaded.ndim == 2 and loaded.shape[1] == 2, f"unexpected shape: {loaded.shape}"
assert np.all(np.diff(loaded[:, 0]) > 0), "timestamps are not strictly increasing"

print(f"shape:      {loaded.shape}")
print(f"time range: {loaded[0, 0]:.3f} s → {loaded[-1, 0]:.3f} s")
print("OK")